In [26]:
import os
import cv2
import json
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetV2S
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Input, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

*CNN with MobileNetV2*

In [9]:
def build_mobilenet_v2():
    base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = False 
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    return finalize_research_model(inputs, x, "MobileNetV2")

*CNN with EfficientNetV2S*

In [10]:
def build_efficientnet_v2s():
    base = EfficientNetV2S(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = False
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    return finalize_research_model(inputs, x, "EfficientNetV2S")

*CNN with ResNet50*

In [11]:
def build_resnet_50():
    base = ResNet50(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = False
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    return finalize_research_model(inputs, x, "ResNet50")

In [22]:
def build_benchmark_model(model_type, input_shape=(224, 224, 3)):
    inputs = Input(shape=input_shape)
    
    if model_type == "MobileNetV2":
        base = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    elif model_type == "ResNet50":
        base = ResNet50(input_shape=input_shape, include_top=False, weights='imagenet')
    elif model_type == "EfficientNetV2S":
        base = EfficientNetV2S(input_shape=input_shape, include_top=False, weights='imagenet')
    
    base.trainable = False
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    # Standard 16-D Bottleneck for feature isolation
    x = Dense(16, activation="relu")(x)
    outputs = Dense(1, activation="sigmoid")(x)
    
    model = Model(inputs, outputs, name=model_type)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return model

In [ ]:
def check_comparative_performance():
    # 1. Path Setup (Relative to notebook)
    tabular_dir = "training dataset/tabular"
    image_dir = "training dataset/image"
    
    # Load the split files
    train_df = pd.read_csv(os.path.join(tabular_dir, "train_split.csv"))
    val_df = pd.read_csv(os.path.join(tabular_dir, "val_split.csv"))
    test_df = pd.read_csv(os.path.join(tabular_dir, "test_split.csv"))
    
    # Find image path columns
    view_cols = [c for c in train_df.columns if "View" in c and "Path" in c]
    
    # 2. Image Loading Logic (Generator)
    def data_generator(df):
        for _, row in df.iterrows():
            if row.get('Has_Images', 0) == 1:
                for col in view_cols:
                    p = row[col]
                    if pd.notna(p) and p != "MISSING_IMAGE":
                        # Adjust path for OS compatibility
                        p = p.replace('\\', os.sep).replace('/', os.sep)
                        full_path = os.path.join(image_dir, p)
                        
                        img = cv2.imread(full_path)
                        if img is not None:
                            img = cv2.resize(img, (224, 224))
                            yield img / 255.0, row['Diagnosis']

    def get_ds(df, shuffle=False):
        ds = tf.data.Dataset.from_generator(
            lambda: data_generator(df),
            output_signature=(tf.TensorSpec((224, 224, 3), tf.float32), tf.TensorSpec((), tf.int32))
        )
        if shuffle: ds = ds.shuffle(200)
        return ds.batch(32).prefetch(tf.data.AUTOTUNE)

    # 3. Benchmark Loop
    models = ["MobileNetV2", "ResNet50", "EfficientNetV2S"]
    results = {}

    for m_name in models:
        print(f"\n🚀 Testing Backbone: {m_name}")
        model = build_benchmark_model(m_name)
        train_ds = get_ds(train_df, shuffle=True)
        val_ds = get_ds(val_df)
        test_ds = get_ds(test_df)
    
        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=10,
            verbose=1,
            callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=3, mode='max')]
          )
        
        loss, acc, auc = model.evaluate(test_ds, verbose=0)
        
        y_true = []
        y_pred = []

    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        
        # If binary classification
        preds = (preds > 0.5).astype(int).flatten()
        
        y_true.extend(labels.numpy())
        y_pred.extend(preds)

    # 🔥 Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    
    print(f"\n📊 Confusion Matrix for {m_name}:")
    print(cm)

    print(f"\n📋 Classification Report for {m_name}:")
    print(classification_report(y_true, y_pred))

    results[m_name] = {
        "Accuracy": round(acc, 4),
        "AUC": round(auc, 4),
        "TP": cm[1][1],
        "TN": cm[0][0],
        "FP": cm[0][1],
        "FN": cm[1][0]
    }

    print(f"✅ {m_name} Test AUC: {auc:.4f}")

    return pd.DataFrame(results).T

# --- Run the Benchmark ---
performance_report = check_comparative_performance()
print(performance_report)


🚀 Testing Backbone: MobileNetV2
Epoch 1/10
     38/Unknown 25s 517ms/step - accuracy: 0.7177 - auc: 0.5391 - loss: 0.6368

C:\Users\Trinabh Mitra\AppData\Roaming\Python\Python313\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


38/38 ━━━━━━━━━━━━━━━━━━━━ 31s 677ms/step - accuracy: 0.7357 - auc: 0.5848 - loss: 0.6133 - val_accuracy: 0.7322 - val_auc: 0.7063 - val_loss: 0.5623
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 23s 582ms/step - accuracy: 0.7804 - auc: 0.7411 - loss: 0.4829 - val_accuracy: 0.8087 - val_auc: 0.7109 - val_loss: 0.4970
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 22s 546ms/step - accuracy: 0.8078 - auc: 0.8039 - loss: 0.4205 - val_accuracy: 0.7814 - val_auc: 0.7178 - val_loss: 0.4801
Epoch 4/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 20s 504ms/step - accuracy: 0.8360 - auc: 0.8603 - loss: 0.3711 - val_accuracy: 0.8033 - val_auc: 0.7404 - val_loss: 0.4615
Epoch 5/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 20s 507ms/step - accuracy: 0.8509 - auc: 0.8857 - loss: 0.3451 - val_accuracy: 0.7760 - val_auc: 0.6949 - val_loss: 0.4627
Epoch 6/10
27/38 ━━━━━━━━━━━━━━━━━━━━ 4s 440ms/step - accuracy: 0.8807 - auc: 0.8898 - loss: 0.3129